In [ ]:
#| code-fold: true
#| code-summary: imports and setup

In an [earlier post](https://matttriano.dev/posts/021_bike_map/bike_map_routing_tool.html) I described a bike router that picks routes by how stressful they are to ride rather than how short they are — how it turns OpenStreetMap edges, Chicago crash data, and bike infrastructure into a per-segment cost, and runs A\* pathfinding over that cost in an AWS Lambda function. This post is about why I rewrote that Lambda in Rust, a language I'd barely used, and what the rewrite taught me about when reaching for performance is worth it.

In Chicago, the Python version worked well enough for most of the realistic routes, but past 20 to 25 miles, I could feel the strain while I waited several seconds for the route to appear. I wanted to take on bigger areas, like the San Francisco Bay Area and New York City, but it was out of the question at the prior level of performance. For SF Bay or NYC-sized graphs, the Python Lambda Function would have taken long enough per route that a first-time visitor would assume the tool was broken and click away before it answered. And since Lambda bills by the gigabyte-second, slow routes on big graphs would have pushed me out of the free tier if usage ever picked up — though that was nearly moot, because nobody waits around for a router that looks broken. I was optimizing against a wall I could see coming, not a fire already burning.

## Python is the fastest way to get it working

The hard part of this project was never the routing algorithm in the abstract. A\* is A\*. The hard part was turning OpenStreetMap's messy, inconsistent tags into a stress cost and tuning the search until the routes looked right — a loop you run hundreds of times, eyeballing maps and adjusting weights.

Python is unbeatable for that loop. The ecosystem handed me the geospatial and graph tooling for free, and because it's interpreted I could work interactively and check each piece as I built it. For most data engineering and data science work this is the whole game, and it's worth saying plainly: unless you've built something badly, there usually isn't much performance left to win, and reaching for a faster language by default is its own mistake. The Python prototype wasn't a draft I'd come to regret. It was where all the real thinking happened, and I'd make the same call again.

## Until "fast enough" becomes a requirement

Sometimes performance is the requirement. "Route across the Bay Area fast enough that a user doesn't give up" is a system goal, and no amount of careful Python was going to meet it, because the cost was structural.

The graph shipped as a gzipped, pickled NetworkX object, and two problems followed from that. Cold start meant reconstructing every nested Python object on load, which meant it took 8 to 15 seconds before the function could answer its first request, after loading 1200 MB to 1600 MB worth of bits into memory. And warm routing paid Python's per-object overhead on every node the search touched, so compute time grew superlinearly with route length: barely perceptable across a neighborhood, but impossible to ignore at the city-wide scale.

You can always find ways to make your code faster or use less memory, if you're:
1. willing to put in the effort,
2. see some inefficient step that you can improve, and
3. accept the risk of introducing new bugs (and the future work of fixing them).
Optimizing can have some real costs, so you have to way the potential benefit * chance of success against the cost and walk away if the juice isn't worth the squeeze. It doesn't make sense to spend days to make a process you run a few times a few percent faster. But if the performance gain is large enough to go from "only serve small cities or make a product is so slow it seems broken" and "the product handles giant cities without breaking a sweat or much cost", it's worth trying to find a design that delivers that level of performance. I ruled out the cheap moves first: tuning the memory allocation for more vCPU, a leaner in-memory graph, keep-warm pings. They nibbled at the edges but didn't touch the structural costs.

What I kept coming back to was the graph itself. I knew Rust was fast, but the insight that sold me was that its type system would let me represent the graph far more compactly than a gzipped dict-of-dicts — a packed, fixed-layout binary instead of a pile of Python objects. That compactness was the lever: a smaller graph attacks the cold start and the package size at the same moment the language change attacks the compute. So I in asked the question I'd been circling — how feasible is it to reimplement this in Rust, given that I've written almost none of it?

## The prototype as a test oracle

What made this a ten-hour job rather than a month-long one is that I wasn't starting from a blank page. I already had a correct, well-tested Python implementation, so the rewrite was a translation problem with a built-in answer key: whatever route the Rust version produced, I could check against the Python version on the same coordinates. The scope was bounded to one service, one algorithm, and an interface I had to preserve exactly.

So I built it in ten focused chunks — the workspace skeleton, the packed binary graph format and its reader, the A\* core, a small KD-tree for nearest-node lookup, the Lambda glue, the deployment cutover — and validated after each one against the known-good Python behavior rather than building the whole thing and hoping at the end. That cadence is the actual lesson. Validating each chunk meant unexpected behavior surfaced early, while it was cheap to fix, instead of compounding onto a bad foundation or slipping through as a silent bug — and a wrong routing weight is exactly the kind of bug that produces a plausible-looking but subtly worse route without ever throwing an error. One example of the discipline: the tests for the binary format deliberately don't round-trip through my own writer, because a shared misunderstanding between reader and writer would round-trip cleanly while being wrong; they assert against hand-computed byte layouts instead. There was plenty of real friction — pinning the exact Rust toolchain the AWS SDK demanded, working around a dependency whose CPU intrinsics wouldn't cross-compile to ARM — and doing it chunk by chunk let me adapt as I hit each one. (I leaned on an AI assistant to write much of the unfamiliar Rust; the judgment about what to build, what to verify, and when to throw an approach out was the part that made it work.)

### The complexity I chose not to build

One decision is worth calling out because it never shows up in a benchmark. Rust needs a compile-and-package step Python doesn't, and I spent an hour exploring how to wire that build into the Airflow DAG that refreshes each city. Then I threw it out. The binary only needs rebuilding when I change the Rust, which is rare — so paying that cost on every city refresh, and taking on the orchestration to automate it, would have been effort against a problem I didn't have. A Makefile that makes a manual build a one-liner was the right tool. Knowing which complexity to refuse is most of the job.

In [3]:
0.046 * 50 ** 1.49

15.639509857552161

## The difference

I benchmarked both implementations identically: the same 50 origin-destination pairs, ten in each of five distance bands (under 5 miles, 5–10, 10–20, 20–35, and 35–50), run through each function. Fitting compute time against route length with `np.polyfit`:

- python: $t \approx 0.046\,x^{1.49}$
- rust: $t \approx 0.058\,x^{0.94}$

![Compute time versus route length for the Python and Rust routing Lambdas. Python scales superlinearly; Rust scales nearly linearly.](imgs/rust_v_python_routing_times.png)

This is *not* a complexity improvement, and it's worth being precise about that: it's the same A\*, the same graph, the same heuristic, expanding a similar set of nodes for a given route. What changed is the cost of touching each node. Python pays object overhead and dict-of-dicts pointer-chasing on every visit, and as longer routes grow the search frontier those costs compound — more allocation, more cache misses chasing pointers through a graph that doesn't live in contiguous memory — so wall-clock time inflates faster than the work itself. Rust's graph is a flat, contiguous structure indexed by integer, so per-node cost stays essentially constant and the measured time tracks the actual work. Note the honest wrinkle: Rust's coefficient is slightly *higher*, so on very short routes the two are neck-and-neck. Short routes were never the problem. The whole win lands on exactly the long, cross-metro routes that made big cities impossible — and it widens with every mile.

The other numbers tell the same before-and-after story. Cold start dropped from 8–15 seconds to 1–2. The deployment package went from 67 MB to 4.9 MB. The function's real memory footprint fell from a max of 1200–1600 MB to 200–400 MB — though I now deliberately allocate 1024 or 1536 MB by city, not because I need the memory but because more memory buys more vCPU, and the work is finally compute-bound rather than dominated by loading the graph. It used to be I/O-bound and bloated; now it's lean and fast.

And here's the payoff I couldn't reach before: open the [San Francisco Bay Area map](https://sf.bikeinfra.com/) and ask for a route so long you'd need two days to actually bike it. Watch how fast it answers. That route simply wasn't servable on the old implementation, and being able to serve it is the entire reason I did this.


The reimplemented rust router took just over 3 seconds to calculate this 107.6 mile (or 173.1 kilometer) route.
The measurements on the plot above show that the Python implementation took a bit over 15 seconds to calculate a 50 mile route. By the equation on that plot, the Python implementation would have taken 49 seconds (!!!) to calculate the same route, and the Python implementation had 33% more memory allocated to it, so it is nearly 22x as expensive to calculate that route via the Python implementation instead of via the Rust implementation. That's not an incremental improvement; that's a categorical, order of magnitude improvement in performance that makes this platform into a viable tool instead of a neat proof of concept.

In [4]:
0.046 * 107.6 ** 1.49

48.99571682822895

In [9]:
(49 / 3) * 1.33

21.723333333333333

![](imgs/long_sf_route.jpeg)
![](imgs/long_sf_route_cw_cost.jpeg)

## If you're weighing the same swap

Prototype in the language that lets you think; move the hot path to the language that lets it run — but only the hot path, only once the prototype is correct enough to be your test oracle, and only carrying the complexity you actually need. The reason this was ten hours and not a month is that the hard part — knowing what to build — was finished before I wrote a line of Rust.